# STAT 301 Project — Online Shoppers Purchasing Intention 


**Name:** Samika - 27276179, Shrey - 34594788, Ivy - 39512314, Wilson - 92153378  \
**Group:**  Group 25 \
**Date:** Dec. 6th \
**Class:** STAT301

---

## Introduction

### 1. Background 
Research on online consumer behavior shows that browsing patterns—such as time spent on pages and navigation sequences—offer strong signals of purchase intention (Moe & Fader, 2004). Prior studies also find that engagement metrics like bounce and exit behavior are closely related to transaction likelihood (Bucklin & Sismeiro, 2003), motivating our examination of how behavioral and contextual factors in this dataset are associated with purchase outcomes.In this project, we investigate which behavioural and contextual factors—such as the number and type of pages viewed, the value of pages visited, exit behaviour, and visit characteristics—are most strongly associated with the probability of completing a purchase. 

### 2. Covariates for Response
Our response variable is Revenue, a binary indicator of whether the session resulted in a purchase (“Purchase”) or not (“No Purchase”). To explore behavioural factors that may influence purchasing likelihood, we focus on predictors that capture three major dimensions of online activity:
1. **Browsing depth** (*Administrative*, *Informational*, *ProductRelated*)
2. **Engagement quality** (*ExitRates*, *PageValues*)
3. **Session timing and visitor profile** (*SpecialDay*, *Weekend*, *VisitorType_simple*)

### 3. Research Question
How are key aspects of a user’s browsing depth, exit behaviour, timing and profile are associated with the probability that an online shopping session results in a purchase?

### 4. Aim and Goal
The primary aim of this project is **inference**. Rather than building a model to predict purchases for new sessions, we are interested in understanding **how** the selected behavioural and timing variables are associated with the odds of making a purchase. Our goal is to quantify and interpret these associations (for example, how changes in `PageValues` or `ExitRates` relate to the odds of a purchase), while recognizing that the observational nature of the data limits causal conclusions.

## Data Description

### 1. Descriptive Summary

This data contains information about user behavior when online shopping. Each row in the dataset represents one online shopping session, and the variables describe various aspects of user behavior, technical attributes, and timing information. The dataset includes both quantitative and categorical variables, such as the number and duration of pages visited, bounce and exit rates, month of visit, type of operating system, and whether the session occurred on a weekend.  

- **Number of observations:** 12330  
- **Number of variables:** 18
#### Variable Description
  | Variable Name | Type | Description |
|-----------------------------|------------------|-------------------------------------------------------------|
| Administrative |Numeric(Integer) | Number of administrative pages visited by the user. |
| Informational | Numeric(Integer) | Number of informational pages visited by the user. |
| ProductRelated | Numeric(Integer) | Number of product-related pages visited by the user. |
| BounceRates | Numeric(Continuous) | Percentage of visitors who enter the site from that page and then leave ("bounce") without triggering any other requests to the analytics server during that session. |
| ExitRates | Numeric(Continuous) | Percentage of this page being the last session. |
| PageValues | Numeric(Integer) | Average value for a web page that a user visited before completing an e-commerce transaction.  |
| SpecialDay | Numeric(Integer) | Closeness of the site visit date to a special day (e.g., Mother’s Day, Christmas). Ranges from 0 to 1. |
| VisitorType | Categorical | Indicates if the visitor is a Returning Visitor or New Visitor. |
| Weekend | Binary (True/False) | Indicates whether the visit occurred on a weekend. |
| Revenue | Binary (0/1) | Indicates whether the session ended with a purchase (1 = purchase, 0 = no purchase). |

---


### 2. Source and Information 
This dataset is the **Online Shoppers Purchasing Intention** dataset published by UCI Machine Learning Repository. 
It contains session-level features for e-commerce website visits and a binary outcome `Revenue` indicating whether a purchase occurred. 

This dataset is **observational** rather than experimental. It consists of naturally occurring online shopping sessions without any random assignment or intervention, meaning all variables reflect real user behavior and session characteristics as they occurred.

### 3. Variable Transformations
To prepare the dataset for modelling, rows with missing or invalid `Region` values were removed, and the response variable Revenue was converted to a labelled factor. The `Weekend` variable was simplified into “Weekday” and “Weekend,” and `VisitorType` was consolidated into “Returning Visitor,” “New Visitor,” and “Other” to avoid sparse categories. These transformations ensure variables are in appropriate formats and improve the interpretability and robustness of the final regression model.

In [ ]:
# loading the appropriate libraries and packages
library(tidyverse) 
library(broom)
library(car) 
library(forcats)
library(patchwork)

In [ ]:
data <- read_csv("https://raw.githubusercontent.com/Ivvvyyan/STAT-group25/refs/heads/main/online_shoppers_intention.csv") # loaded dataset from URL

data <- data |>
        filter(!is.na(Region), Region != 1) |>
        mutate(Revenue = factor(Revenue,
                                levels = c(FALSE, TRUE),
                                labels = c("No Purchase", "Purchase")),
               NearSpecialDay = if_else(SpecialDay > 0, "Near special shopping day", "Regular day"),
               Weekend = factor(Weekend,
                                levels = c(FALSE, TRUE),
                                labels = c("Weekday", "Weekend")),
               VisitorType_simple = fct_lump_n(VisitorType, n = 2, other_level = "Other"),
               VisitorType_simple = fct_relevel(VisitorType_simple, "Returning_Visitor"))

### 4. Variables redundant and selection

#### Response Variable

The response variable is **Revenue**, which indicates whether the online shopping session ended with a purchase:

- `Revenue = "Purchase"`: the session resulted in a purchase.  
- `Revenue = "No Purchase"`: the session did not result in a purchase.

This binary outcome makes logistic regression an appropriate modelling choice.

#### Explanatory Variable Selection
To decide which explanatory variables to include in the final model, in stage 2, we first considered a set of numeric predictors that capture browsing depth, exit behaviour, and timing: `Administrative`, `Administrative_Duration`, `Informational`, `Informational_Duration`, `ProductRelated`, `ProductRelated_Duration`, `BounceRates`, `ExitRates`, `PageValues`, and `SpecialDay`. We fitted an initial logistic regression using all of these numeric variables and examined the coefficient patterns, p-values, a correlation matrix, and variance inflation factors (VIFs).

The correlation matrix showed that each page-count variable was strongly correlated with its corresponding duration variable (for example, `ProductRelated` vs. `ProductRelated_Duration`). Because these pairs describe the same underlying behaviour, we kept the page-count variables (`Administrative`, `Informational`, `ProductRelated`) and removed their duration counterparts to avoid redundancy and multicollinearity. A moderate correlation was also observed between `BounceRates` and `ExitRates`; we retained `ExitRates` as a broader measure of page exits and dropped `BounceRates`. In contrast, `PageValues` and `SpecialDay` had low correlations with other predictors and VIFs close to 1, indicating that they provide unique explanatory information.

After these checks, the final numeric set used in our model is:
`Administrative`, `Informational`, `ProductRelated`, `ExitRates`, `PageValues`, and `SpecialDay`. We then added the categorical predictors `Weekend` and `VisitorType_simple` to account for timing (weekday vs. weekend sessions) and visitor profile (returning vs. new or other visitors). All variables in the final model have VIF values below 5, suggesting that multicollinearity is not a major concern.



In [ ]:
# Candidate numeric predictors considered in Stage 2
cand_num_vars <- c(
  "Administrative", "Administrative_Duration",
  "Informational", "Informational_Duration",
  "ProductRelated", "ProductRelated_Duration",
  "BounceRates", "ExitRates",
  "PageValues", "SpecialDay"
)

# Fit the initial numeric-only logistic regression model
full_formula_num <- as.formula(
  paste("Revenue ~", paste(cand_num_vars, collapse = " + "))
)

full_glm_num <- glm(full_formula_num, data = data, family = "binomial")

# ----- Table 1: Coefficient estimates -----
summary(full_glm_num)

# ----- Table 2: Correlation matrix for numeric predictors -----
corr_matrix <- data %>%
  select(all_of(cand_num_vars)) %>%
  cor()

corr_matrix

# ----- Table 3: VIFs for the numeric-only model -----
full_vif_num <- vif(full_glm_num)
full_vif_num

## EDA Visualisations 

In [ ]:
#production exploration vs Exit rate plot

x_limit <- quantile(data$ProductRelated, 0.99, na.rm = TRUE)
y_limit <- quantile(data$ExitRates, 0.99, na.rm = TRUE)

eda_plot <- ggplot(data, aes(x = ProductRelated, y = ExitRates, colour = Revenue)) +
  geom_point(alpha = 0.25, size = 1) +
  geom_smooth(
    method = "loess",
    se = FALSE,
    linewidth = 1) +
  facet_grid(
    rows = vars(NearSpecialDay),
    cols = vars(VisitorType_simple)) +
  coord_cartesian(
    xlim = c(0, x_limit),
    ylim = c(0, y_limit)) +
  labs(
    title = "Product Exploration vs Exit Rates by Purchase Outcome",
    subtitle = "Faceted by special shopping days and visitor type",
    x = "Number of product-related pages viewed (ProductRelated)",
    y = "Average exit rate from visited pages (ExitRates)",
    colour = "Purchase outcome") +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    plot.subtitle = element_text(size = 11),
    strip.text = element_text(face = "bold"),
    legend.position = "bottom",
    panel.grid.minor = element_blank()) +
  guides(
    colour = guide_legend(
      override.aes = list(alpha = 1, size = 3))
  )

In [ ]:
# boxplot of page duration and revenue with type of durations

options(repr.plot.width = 10, repr.plot.height = 4)
data_duration_visit_df <- data %>%
    select(VisitorType_simple, Revenue, ProductRelated_Duration, Administrative_Duration, Informational_Duration) %>%
    pivot_longer(cols = c(ProductRelated_Duration, Administrative_Duration, Informational_Duration), # combine all page durations to one group
                 names_to = "DurationType",
                 values_to = "DurationValue")

data_duration_visit_plot <- data_duration_visit_df %>%
    ggplot() + 
    geom_jitter(aes(x = as.factor(Revenue), y = DurationValue, colour = VisitorType_simple), alpha = 0.5, width = 0.2) +
    geom_boxplot(aes(x = as.factor(Revenue), y = DurationValue), alpha = 0.5) + 
    facet_wrap( ~ DurationType) + # separate boxplots according to type of duration
    scale_y_log10() + # code to improve readibility
    ggtitle("Probability of Revenue and Web Page Duration") + 
    labs(x = "Revenue",
         y = "Web Page Duration (in log10)") +
    theme(
        text = element_text(size = 15),
        plot.title = element_text(face = "bold"),
        axis.title = element_text(face = "bold")
    )

eda_plot #print the plot above
data_duration_visit_plot # print this plot 

## Methodology

### 1. Step-by-step procedure

1. **Load and preprocess the data.**  
   We started from the provided `data` object (Online Shoppers Purchasing Intention dataset). We removed sessions with missing or invalid `Region` values, converted `Revenue` into a labelled factor (“No Purchase”, “Purchase”), recoded `Weekend` into “Weekday” vs “Weekend”, and simplified `VisitorType` into `VisitorType_simple` with three levels (“Returning_Visitor”, “New_Visitor”, “Other”) using factor lumping. These transformations ensured that the response and categorical predictors were in the correct format for logistic regression and that sparse categories were combined.

2. **Exploratory data analysis.**  
   We created a main visualization of `ProductRelated` (number of product-related pages) versus `ExitRates`, coloured by `Revenue` and faceted by special-day indicator and visitor type. This plot allowed us to visually assess how product exploration and exit behaviour differ between purchase and non-purchase sessions across different types of visitors and timing.

3. **Initial numeric-only logistic regression and diagnostics (Stage 2).**  
   Focusing on numeric predictors only, we fitted a full logistic regression model with `Revenue` as the response and all candidate numeric variables (page counts, durations, exit rates, page values, and special-day proximity) as predictors. From this model, we examined coefficient estimates and p-values, computed a correlation matrix for the numeric predictors, and calculated variance inflation factors (VIFs) to diagnose multicollinearity.

4. **Selection of numeric explanatory variables (Stage 2).**  
   Using the coefficient patterns, correlations, and VIFs, we removed redundant predictors. Duration variables were dropped because they were highly correlated with their corresponding page-count variables (e.g., `ProductRelated` vs `ProductRelated_Duration`) and did not add distinct explanatory power. Between `BounceRates` and `ExitRates`, we retained `ExitRates` because it provides a broader measure of page exits and had acceptable VIFs. The final numeric set consisted of `Administrative`, `Informational`, `ProductRelated`, `ExitRates`, `PageValues`, and `SpecialDay`.

5. **Build the final logistic regression model.**  
   We then constructed a combined logistic regression model including both the selected numeric predictors and the categorical predictors `Weekend` and `VisitorType_simple`. The response variable remained `Revenue` (Purchase vs No Purchase). This final model relates the log-odds of a purchase to browsing depth (page counts), exit behaviour, the value of visited pages, timing relative to special days, and visitor profile, using the `glm()` function with a binomial family.

6. **Model diagnostics for the final model.**  
   For the final model, we re-computed VIFs to confirm that multicollinearity is not severe (all VIFs < 5). We also test its Overdispersion and use the Residual plot and the Q-Q plot.

7. **Results summarisation and interpretation.**  
   Finally, we extracted the estimated coefficients and p-values, and also exponentiated the coefficients to obtain odds ratios. We focused our interpretation on statistically significant predictors (e.g., `PageValues`, `ExitRates`, `SpecialDay`, and visitor type), explaining in plain language how changes in each variable are associated with the odds of a purchase while holding other variables fixed. These results were then linked back to the original scientific question about how browsing behaviour, exit behaviour, and visit timing relate to purchasing.

### 2. Justification

This methodology follows the plan from Stage 2. Because `Revenue` is a binary outcome, logistic regression is an appropriate choice for modelling how explanatory variables affect purchase probability. Starting with exploratory visualisation helps us understand key relationships and potential non-linearities. The staged variable-selection process—fitting a full numeric model, examining correlations and VIFs, and then adding categor¥ical predictors—ensures that the final model includes non-redundant, interpretable predictors while controlling for multicollinearity. Checking VIFs and comparing null vs residual deviance provides basic model diagnostics consistent with the course. The final step of interpreting coefficients and odds ratios directly addresses our inferential goal: understanding which aspects of user behaviour and timing are associated with online purchases.

### 3. Assumptions

The main assumptions underlying this analysis are:

- **Independent observations:** Each session is treated as an independent visit; one user’s behaviour does not directly affect another’s within the dataset.
- **Logistic regression form:** The log-odds of a purchase are assumed to be a linear function of the predictors included in the model.
- **No severe multicollinearity:** Predictors are not highly collinear, which we checked using correlation matrices and VIFs.
- **Correct model specification:** The chosen predictors capture the main behavioural and timing effects relevant to purchase outcome, and the logit link is appropriate for a binary response.

## Code and Result

### 1. Justification

In [ ]:
#baseline model
model1 <- glm(
  Revenue ~ Administrative,
  data = data,
  family = "binomial"
)

#full model
model_full <- glm(
  Revenue ~ Administrative + Informational + ProductRelated +
    ExitRates + PageValues + SpecialDay +
    Weekend + VisitorType_simple,
  data = data,
  family = "binomial"
)

In [ ]:
#anova test
anova(model1, model_full)

we observe a substantial reduction in residual deviance when moving from Model 1 to the full model, indicating that the additional predictors collectively improve model fit. So we choose our full model. 

### 2. Final Model
Let \(Y\) be the indicator that the session results in a purchase,  
so \(Y = 1\) if `Revenue = "Purchase"` and \(Y = 0\) if `Revenue = "No Purchase"`.

The logistic regression model is

log-odds(Purchase) =
β₀
+ β₁·Administrative
+ β₂·Informational
+ β₃·ProductRelated
+ β₄·ExitRates
+ β₅·PageValues
+ β₆·SpecialDay
+ β₇·I(Weekend = "Weekend")
+ β₈·I(VisitorType_simple = "New_Visitor")
+ β₉·I(VisitorType_simple = "Other").

Here:

- `Weekend = "Weekday"` is the baseline level for **Weekend**.
- `VisitorType_simple = "Returning_Visitor"` is the baseline level for **VisitorType_simple**.

Each coefficient \(\beta_j\) describes the change in the log-odds of a purchase associated with a one-unit increase in the corresponding numeric predictor, or with switching from the baseline category to the indicated level for categorical predictors, holding all other variables fixed.

In [ ]:
tidy(model_full, conf.int=TRUE)

### 3. Model Diagnostics

In [ ]:
#Overdispersion Problem in Logistic Regression
data_logistic_quasi <- glm(Revenue ~ Administrative + Informational + ProductRelated + # method of getting dispersion parameter with quasibinomial
    ExitRates + PageValues + SpecialDay +
    Weekend + VisitorType_simple, family = quasibinomial, data = data)

summary(data_logistic_quasi)$dispersion # show only the dispersion parameter

The dispersion parameter has a value of about 17075.59

The overdispersion parameter is much larger than 1, indicating the variance observed in the data is greater than the variance computed by the model. Since the overdispersion parameter is large, the model underestimates the standard error and computes a larger test statistic, which provides us with an unreliable p-value for hypothesis tests.

In [ ]:
#Test VIF again for final model
vif_final <- vif(model_full)
vif_final

In [ ]:

fitted_vals <- fitted(model_full, type = "response")   # predicted probability
residual_vals <- residuals(model_full, type = "deviance")

diag_df <- data.frame(
  Fitted = fitted_vals,
  Residuals = residual_vals
)

# Residuals vs Fitted Plot
p_resid <- ggplot(diag_df, aes(x = Fitted, y = Residuals)) +
  geom_point(color = "steelblue", alpha = 0.6) +
  geom_hline(yintercept = 0, colour = "red", linetype = "dashed") +
  labs(
    title = "Residuals vs Fitted",
    x = "Fitted Values (Predicted Probability)",
    y = "Deviance Residuals"
  ) +
  theme_minimal()


# Draw QQ graph
std_resid <- rstandard(model_full)   # standardized residuals

p_qq <- ggplot(data.frame(std_resid), aes(sample = std_resid)) +
  stat_qq(color = "steelblue", alpha = 0.6) +
  stat_qq_line(colour = "red",linetype = "dashed") +
  labs(
    title = "Q-Q Plot of Residuals",
    x = "Theoretical Quantiles",
    y = "Sample Quantiles"
  ) +
  theme_minimal()

combined_plot <-(p_resid + p_qq) +
  plot_annotation(
    title = "Model Diagnostic Plots",
    theme = theme(plot.title = element_text(size = 16, face = "bold"))
  )

# Draw Coefficient plot
coef_data <- tidy(model_full, conf.int = TRUE) |>
  filter(term != "(Intercept)") |>
  mutate(term = fct_reorder(term, estimate))

p_coef <- ggplot(coef_data, aes(x = estimate, y = term)) +
  geom_point(size = 2, colour = "darkblue") +
  geom_errorbarh(
    aes(xmin = conf.low, xmax = conf.high),
    height = 0.2,
    colour = "darkblue"
  ) +
  geom_vline(xintercept = 0, linetype = "dashed", colour = "red") +
  labs(
    title = "Coefficient Estimates with 95% Confidence Intervals",
    x = "Coefficient Estimate",
    y = "Predictor Variable"
  ) +
  theme_minimal()
print(combined_plot)
print(p_coef)

## Discussion

### 1. Interpretation
Our results show that deeper engagement with product-related pages and higher PageValues significantly increase the probability of purchase, while high exit rates strongly reduce it. Weekend sessions and returning visitors are also more likely to convert, whereas visits occurring closer to special days are less likely to lead to a purchase. Overall, the findings suggest that both browsing behavior and user characteristics offer meaningful signals about a session’s purchase likelihood and can inform strategies for improving user engagement and conversion.

### 2. Limitation
This study has several limitations. First, the model diagnostics show that logistic regression does not fully capture the data structure: the residual plot displays a curved pattern and the Q-Q plot shows heavy tails, suggesting nonlinearity and deviations from model assumptions. Second, the outcome variable is highly imbalanced, which leads the model to produce fitted probabilities very close to 0 or 1 in some cases; this can make coefficient estimates unstable. Third, because the dataset is observational and lacks random assignment, we can only identify associations rather than causal effects.Finally, the data come from one specific e-commerce website, so the results may not generalize to other platforms or time periods.

### 3. Improvement
Some methods to improve a logistic model are handling outliers or computing a more complex logistic model. When we do not address outliers, they can inflate the estimated coefficients of the association and lead to incorrect p-values. If there is suspicion that the effect of one covariate affects another, we can implement a logistic model with interaction terms to better fit the data. However, creating a model with interaction terms makes interpretations of the model more complicated.

### 4. Future Questions
Incorporating additional behavioral or contextual variables may help explain purchasing determinants that the current model fails to capture.

## References

Sakar, C. & Kastro, Y. (2018). Online Shoppers Purchasing Intention Dataset [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5F88Q.

Moe, W. W., & Fader, P. S. (2004). Capturing evolving visit behavior in clickstream data. Journal of Interactive Marketing, 18(1), 5–19. 
https://doi.org/10.1287/mnsc.1040.0153

Bucklin, R. E., & Sismeiro, C. (2003). A model of web site browsing behavior estimated on clickstream data. Journal of Marketing Research, 40(3), 249–267.    
https://doi.org/10.1509/jmkr.40.3.249.19